# 基于深度强化学习的期货量化交易策略 (PPO)

**论文参考**: 陈雪梅 (2023), 《基于深度强化学习的期货量化交易策略研究》, 盈利率83.3%

**核心思想**: PPO适配期货市场，支持做空操作。动作空间为 {-1: 做空, 0: 空仓, 1: 做多}，引入保证金账户机制。

**数据**: 沪深300指数 (sh000300) 作为期货代理

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 算法原理

### PPO (Proximal Policy Optimization) 回顾

PPO通过截断比率限制策略更新幅度:

$$L^{CLIP}(\theta) = \hat{\mathbb{E}}_t \left[ \min\left( r_t(\theta) \hat{A}_t, \; \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) \hat{A}_t \right) \right]$$

其中 $r_t(\theta) = \frac{\pi_\theta(a_t | s_t)}{\pi_{\theta_{\text{old}}}(a_t | s_t)}$

### 期货市场适配

**双向交易**: 期货允许做空，动作空间扩展为:
$$a \in \{-1, 0, +1\}$$
- $a = +1$: 做多 (买入开仓)
- $a = 0$: 空仓 (平仓)
- $a = -1$: 做空 (卖出开仓)

**保证金机制**: 持仓需要缴纳保证金:
$$\text{margin} = |\text{position}| \times \text{price} \times \text{margin\_rate}$$

**盈亏计算**:
$$\text{PnL} = \text{position} \times (P_t - P_{t-1}) \times \text{multiplier}$$

**奖励函数**: $R_t = \text{position}_{t-1} \times r_t - \text{cost}$

其中 $r_t$ 为日收益率，cost包含手续费和滑点。

In [ ]:
# ============ 动画: 期货盈亏追踪与保证金账户 ============
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

np.random.seed(42)
n_days = 100
prices = 4000 + np.cumsum(np.random.randn(n_days) * 20)
positions = np.zeros(n_days)
# 模拟做多/做空决策
for i in range(1, n_days):
    if prices[i] > prices[i-1] + 15:
        positions[i] = 1
    elif prices[i] < prices[i-1] - 15:
        positions[i] = -1
    else:
        positions[i] = positions[i-1]

pnl = np.zeros(n_days)
margin_account = 100000 * np.ones(n_days)
for i in range(1, n_days):
    daily_pnl = positions[i-1] * (prices[i] - prices[i-1]) * 100
    pnl[i] = pnl[i-1] + daily_pnl
    margin_account[i] = margin_account[i-1] + daily_pnl

frames = []
for step in range(10, n_days + 1, 3):
    colors = ['#4CAF50' if p > 0 else '#F44336' if p < 0 else '#9E9E9E' for p in positions[:step]]
    frames.append(go.Frame(
        data=[
            go.Scatter(x=list(range(step)), y=prices[:step], mode='lines',
                       name='期货价格', line=dict(color='#333', width=2)),
            go.Scatter(x=list(range(step)), y=pnl[:step], mode='lines',
                       name='累计盈亏', line=dict(color='#2196F3', width=2)),
            go.Bar(x=list(range(step)), y=positions[:step],
                   marker_color=colors, name='持仓方向', opacity=0.6),
            go.Scatter(x=list(range(step)), y=margin_account[:step], mode='lines',
                       name='保证金账户', line=dict(color='#FF9800', width=2, dash='dash')),
        ],
        name=f'Day {step}'
    ))

fig = make_subplots(rows=2, cols=2,
                    subplot_titles=['期货价格', '累计盈亏', '持仓方向', '保证金账户'])

fig = go.Figure(
    data=frames[0].data,
    frames=frames,
    layout=go.Layout(
        title=dict(text='期货PPO交易: 盈亏追踪与保证金账户'),
        height=500, width=950,
        xaxis_title='交易日', yaxis_title='值',
        updatemenus=[dict(type='buttons', buttons=[
            dict(label='▶ 播放', method='animate',
                 args=[None, {'frame': {'duration': 100}, 'transition': {'duration': 50}}]),
            dict(label='⏸ 暂停', method='animate',
                 args=[[None], {'frame': {'duration': 0}, 'mode': 'immediate'}]),
        ])],
        sliders=[dict(
            steps=[dict(args=[[f.name], {'frame': {'duration': 0}, 'mode': 'immediate'}],
                        label=f.name, method='animate') for f in frames],
        )],
    )
)
fig.show()

In [ ]:
# ============ 导入与配置 ============
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from shared.data_fetcher import get_index_daily
from shared.backtest_engine import Backtester
from shared.visualization import (plot_equity_curve, plot_drawdown,
                                   plot_metrics_table, set_chinese_font)
from shared.factors import rsi, macd, bollinger_bands, atr, volatility
from shared.constants import *

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# ============ 数据获取与特征工程 ============
df = get_index_daily('sh000300', start_date='20180101', end_date='20241231')
print(f'数据量: {len(df)} 条')

df['returns'] = df['close'].pct_change()
df['rsi'] = rsi(df['close'], 14)
macd_df = macd(df['close'])
df['macd_hist'] = macd_df['histogram']
bb = bollinger_bands(df['close'])
df['bb_pctb'] = bb['bb_pctb']
df['bb_width'] = bb['bb_width']
df['atr_val'] = atr(df['high'], df['low'], df['close'], 14)
vol_df = volatility(df['close'], [5, 10, 20])
df['vol_5'] = vol_df['vol_5']
df['vol_20'] = vol_df['vol_20']
df['mom_5'] = df['close'].pct_change(5)
df['mom_10'] = df['close'].pct_change(10)
df['mom_20'] = df['close'].pct_change(20)

df.dropna(inplace=True)

FEATURE_COLS = ['returns', 'rsi', 'macd_hist', 'bb_pctb', 'bb_width',
                'atr_val', 'vol_5', 'vol_20', 'mom_5', 'mom_10', 'mom_20']
WINDOW = 10
STATE_DIM = len(FEATURE_COLS) * WINDOW
ACTION_DIM = 3  # -1=做空, 0=空仓, 1=做多

# 标准化
feat_mean = df[FEATURE_COLS].mean()
feat_std = df[FEATURE_COLS].std().replace(0, 1)
df[FEATURE_COLS] = (df[FEATURE_COLS] - feat_mean) / feat_std

split = int(len(df) * 0.7)
train_df = df.iloc[:split].reset_index(drop=True)
test_df = df.iloc[split:].reset_index(drop=True)
print(f'训练集: {len(train_df)}, 测试集: {len(test_df)}')

In [ ]:
# ============ 期货交易环境 (支持做空) ============
class FuturesTradingEnv:
    """期货交易环境: 支持做多/做空/空仓"""
    def __init__(self, data, feature_cols, window=10,
                 margin_rate=0.12, commission=0.0001):
        self.data = data.reset_index(drop=True)
        self.feature_cols = feature_cols
        self.window = window
        self.state_dim = len(feature_cols) * window
        self.action_space_n = 3  # 0=空仓, 1=做多, 2=做空
        self.margin_rate = margin_rate
        self.commission = commission
        self.reset()

    def reset(self):
        self.current_step = self.window
        self.position = 0  # -1=做空, 0=空仓, 1=做多
        self.margin_account = 100000.0
        self.initial_capital = 100000.0
        return self._get_state()

    def _get_state(self):
        start = self.current_step - self.window
        end = self.current_step
        features = self.data[self.feature_cols].iloc[start:end].values.flatten()
        return features.astype(np.float32)

    def step(self, action):
        # action: 0=空仓, 1=做多, 2=做空
        action_map = {0: 0, 1: 1, 2: -1}
        target_position = action_map[action]
        current_return = self.data['returns'].iloc[self.current_step]

        # 交易成本
        if target_position != self.position:
            cost = self.commission * 2  # 开仓+平仓
        else:
            cost = 0.0

        # 盈亏 = 仓位 * 收益率
        pnl = self.position * current_return
        reward = pnl - cost

        # 更新保证金账户
        self.margin_account *= (1 + reward)

        self.position = target_position
        self.current_step += 1
        done = self.current_step >= len(self.data) - 1

        # 爆仓检查
        if self.margin_account < self.initial_capital * 0.3:
            done = True
            reward -= 0.1  # 惩罚

        next_state = self._get_state() if not done else np.zeros(self.state_dim, dtype=np.float32)
        return next_state, reward, done, {'margin': self.margin_account}

In [ ]:
# ============ 期货PPO Agent ============
class FuturesActorCritic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(state_dim, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
        )
        self.actor = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, action_dim))
        self.critic = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 1))

    def forward(self, x):
        h = self.shared(x)
        return F.softmax(self.actor(h), dim=-1), self.critic(h)


class FuturesPPOAgent:
    def __init__(self, state_dim, action_dim=3, lr=3e-4, gamma=0.99,
                 eps_clip=0.2, k_epochs=4):
        self.gamma = gamma
        self.eps_clip = eps_clip
        self.k_epochs = k_epochs
        self.action_dim = action_dim

        self.net = FuturesActorCritic(state_dim, action_dim).to(device)
        self.net_old = FuturesActorCritic(state_dim, action_dim).to(device)
        self.net_old.load_state_dict(self.net.state_dict())
        self.optimizer = optim.Adam(self.net.parameters(), lr=lr)

        self.buffer_states = []
        self.buffer_actions = []
        self.buffer_logprobs = []
        self.buffer_rewards = []
        self.buffer_dones = []

    def act(self, state, explore=True):
        s = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            probs, _ = self.net_old(s)
        if explore:
            dist = torch.distributions.Categorical(probs)
            action = dist.sample()
            self.buffer_states.append(state)
            self.buffer_actions.append(action.item())
            self.buffer_logprobs.append(dist.log_prob(action).item())
        else:
            action = probs.argmax(1)
        return action.item()

    def store(self, reward, done):
        self.buffer_rewards.append(reward)
        self.buffer_dones.append(done)

    def train_step(self):
        if len(self.buffer_rewards) == 0:
            return

        # GAE计算
        returns = []
        R = 0
        for r, d in zip(reversed(self.buffer_rewards), reversed(self.buffer_dones)):
            R = r + self.gamma * R * (1 - float(d))
            returns.insert(0, R)

        returns = torch.FloatTensor(returns).to(device)
        if returns.std() > 1e-8:
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        old_states = torch.FloatTensor(np.array(self.buffer_states)).to(device)
        old_actions = torch.LongTensor(self.buffer_actions).to(device)
        old_logprobs = torch.FloatTensor(self.buffer_logprobs).to(device)

        for _ in range(self.k_epochs):
            probs, values = self.net(old_states)
            dist = torch.distributions.Categorical(probs)
            new_logprobs = dist.log_prob(old_actions)
            entropy = dist.entropy()

            ratios = torch.exp(new_logprobs - old_logprobs)
            advantages = returns - values.squeeze().detach()

            surr1 = ratios * advantages
            surr2 = torch.clamp(ratios, 1 - self.eps_clip, 1 + self.eps_clip) * advantages

            loss = (-torch.min(surr1, surr2).mean()
                    + 0.5 * F.mse_loss(values.squeeze(), returns)
                    - 0.01 * entropy.mean())

            self.optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(self.net.parameters(), 0.5)
            self.optimizer.step()

        self.net_old.load_state_dict(self.net.state_dict())
        self.buffer_states = []
        self.buffer_actions = []
        self.buffer_logprobs = []
        self.buffer_rewards = []
        self.buffer_dones = []

In [ ]:
# ============ 训练 ============
N_EPISODES = 100
agent = FuturesPPOAgent(STATE_DIM, ACTION_DIM)
reward_history = []
margin_history = []

for ep in range(N_EPISODES):
    env = FuturesTradingEnv(train_df, FEATURE_COLS, WINDOW)
    state = env.reset()
    ep_reward = 0

    while True:
        action = agent.act(state, explore=True)
        next_state, reward, done, info = env.step(action)
        agent.store(reward, done)
        ep_reward += reward
        state = next_state
        if done:
            break

    agent.train_step()
    reward_history.append(ep_reward)
    margin_history.append(info['margin'])

    if (ep + 1) % 25 == 0:
        print(f'Episode {ep+1}/{N_EPISODES} | Reward: {ep_reward:.4f} | '
              f'Margin: {info["margin"]:,.0f}')

In [ ]:
# ============ 评估与回测 ============
env = FuturesTradingEnv(test_df, FEATURE_COLS, WINDOW)
state = env.reset()
actions_list = [0] * WINDOW
margins = [100000.0] * WINDOW

while True:
    action = agent.act(state, explore=False)
    next_state, _, done, info = env.step(action)
    actions_list.append(action)
    margins.append(info['margin'])
    state = next_state
    if done:
        break

actions_list = actions_list[:len(test_df)]
margins = margins[:len(test_df)]

# 转换信号: 0=空仓, 1=做多, 2=做空
test_dates = df.index[split:split + len(test_df)]
test_prices = df['close'].iloc[split:split + len(test_df)]
test_prices.index = test_dates

sig_map = {0: 0, 1: 1, 2: -1}
signals = pd.Series([sig_map[a] for a in actions_list], index=test_dates[:len(actions_list)])

# 回测 (期货不受T+1约束)
bt = Backtester(t_plus_1=False, commission=0.0001, stamp_tax=0, slippage=0.0005)
result = bt.run(test_prices, signals)

print('期货PPO 策略绩效:')
for k, v in result['metrics'].items():
    print(f'  {k}: {v}')

# 持仓分析
pos_arr = np.array([sig_map[a] for a in actions_list])
print(f'\n持仓分析:')
print(f'  做多比例: {(pos_arr == 1).mean():.2%}')
print(f'  空仓比例: {(pos_arr == 0).mean():.2%}')
print(f'  做空比例: {(pos_arr == -1).mean():.2%}')

In [ ]:
# ============ 可视化 ============
import matplotlib.pyplot as plt
set_chinese_font()

# 1. 训练曲线
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(reward_history, color='#2196F3', alpha=0.5)
axes[0].plot(pd.Series(reward_history).rolling(10).mean(), color='#F44336', linewidth=2)
axes[0].set_title('训练奖励', fontsize=13)
axes[0].set_xlabel('Episode')
axes[0].grid(True, alpha=0.3)

axes[1].plot(margin_history, color='#FF9800')
axes[1].axhline(y=100000, color='gray', linestyle='--', alpha=0.5)
axes[1].set_title('训练期末保证金', fontsize=13)
axes[1].set_xlabel('Episode')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 2. 持仓方向+价格
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True,
                         gridspec_kw={'height_ratios': [2, 1, 2]})

axes[0].plot(test_dates[:len(pos_arr)], test_prices.values[:len(pos_arr)],
             color='#333', linewidth=1.2)
# 标记做多区间(绿)和做空区间(红)
for i in range(1, len(pos_arr)):
    if pos_arr[i] == 1:
        axes[0].axvspan(test_dates[i-1], test_dates[i], color='#4CAF50', alpha=0.15)
    elif pos_arr[i] == -1:
        axes[0].axvspan(test_dates[i-1], test_dates[i], color='#F44336', alpha=0.15)
axes[0].set_title('CSI300指数 (绿=做多, 红=做空)', fontsize=13)
axes[0].set_ylabel('价格')
axes[0].grid(True, alpha=0.3)

# 持仓方向
colors = ['#4CAF50' if p > 0 else '#F44336' if p < 0 else '#9E9E9E' for p in pos_arr]
axes[1].bar(test_dates[:len(pos_arr)], pos_arr, color=colors, alpha=0.7, width=2)
axes[1].set_title('持仓方向', fontsize=13)
axes[1].set_ylabel('方向')
axes[1].set_yticks([-1, 0, 1])
axes[1].set_yticklabels(['做空', '空仓', '做多'])
axes[1].grid(True, alpha=0.3)

# 净值
norm_equity = result['equity_curve'] / result['equity_curve'].iloc[0]
norm_price = test_prices / test_prices.iloc[0]
axes[2].plot(norm_equity.index, norm_equity.values, color='#2196F3',
             linewidth=2, label='期货PPO')
axes[2].plot(norm_price.index[:len(norm_equity)], norm_price.values[:len(norm_equity)],
             color='#9E9E9E', linewidth=1.5, linestyle='--', label='买入持有')
axes[2].set_title('累计净值', fontsize=13)
axes[2].set_ylabel('净值')
axes[2].legend()
axes[2].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 3. 绩效表
plot_metrics_table(result['metrics'], title='期货PPO 绩效指标')

## 实验结果与分析

### 关键发现
1. **双向交易**: 允许做空使策略在下跌行情中也能获利，大幅提高了盈利机会
2. **保证金管理**: 引入保证金账户和爆仓检查，更贴近真实期货交易
3. **PPO稳定性**: 截断比率限制策略更新，在高波动的期货市场中训练更稳定
4. **梯度裁剪**: `clip_grad_norm`进一步防止梯度爆炸

### 期货 vs 股票
- 期货双向交易扩大了策略空间，理论收益上限更高
- 杠杆效应放大收益同时也放大风险
- 无T+1限制，交易更灵活
- 手续费更低(万一左右)

### 改进方向
- 引入动态仓位管理(不仅是方向，还包括仓位大小)
- 加入跨品种套利信号
- 使用真实期货数据(含展期处理)